# Results analysis — P2P-Thief (anrbj666)

Sensitivity analysis (OAT) over the parameters that shape the pursuit game.
Raw data: `results/experiments/sensitivity.json`; regenerate everything with
`uv run python scripts/run_sensitivity.py`.

## 1. Scent decay — why $\rho = 0.10$ is the game's memory

The trail obeys $\tau_t = \tau_0 (1-\rho)^t$ with $\tau_0 = 0.9$. The
lie-detection floor (0.4) is crossed after
$t^* = \frac{\ln(0.4/0.9)}{\ln(1-\rho)}$ turns — at the fixed $\rho=0.10$
that is $t^* \approx 7.7$: a trail stays *legally readable* for ~7 turns,
long enough to catch lies one full round later, short enough that history
does not drown the present (the book's local-optima argument, ch. 4).

![decay](../assets/sens_decay.png)

In [1]:
import json
import math
from pathlib import Path

sens = json.loads(Path("../results/experiments/sensitivity.json").read_text())
for rho in (0.05, 0.10, 0.20, 0.30):
    t_star = math.log(0.4 / 0.9) / math.log(1 - rho)
    print(f"rho={rho}: readable for ~{t_star:.1f} turns")

rho=0.05: readable for ~15.8 turns
rho=0.1: readable for ~7.7 turns
rho=0.2: readable for ~3.6 turns
rho=0.3: readable for ~2.3 turns


## 2. Hint honesty — deception does not save a weak runner

Blind pursuit (belief-only) vs a random thief across honesty levels
$P(\text{truth}) \in \{0, .25, .5, .75, 1\}$, 15 seeds each. Capture rate
stays in the 0.93–1.0 band **independent of honesty**: the scent evidence
dominates the posterior, so lying only helps a thief whose *movement* also
exploits the belief. This validated our decision to invest in movement
strategy over prompt sophistication (moves are pure Python anyway — rule 25).

![honesty](../assets/sens_honesty.png)

In [2]:
print(json.dumps(sens["honesty_capture_rate"], indent=2))

{
  "0.0": 1.0,
  "0.25": 0.8666666666666667,
  "0.5": 0.8666666666666667,
  "0.75": 1.0,
  "1.0": 1.0
}


## 3. Board size — every added cell is thief territory

Mean turns-to-capture under full information grows superlinearly-ish with
the board side (7.7 → 9.3 → 12.1 for 7/9/11): the state space of the
Dec-POMDP grows as $O(n^4)$ (two positions) times barrier configurations,
while the survival threshold stays 35 — so negotiating a larger board is a
pro-thief move, and our cop should resist raising the minimum.

![board](../assets/sens_board.png)

In [3]:
print(sens["board_capture_turns"])

{'7': 7.5, '9': 9.3, '11': 12.1}


## Conclusions

1. Keep $ho=0.10$ (fixed anyway) — the ~7-turn readable window is what
   makes the (1−ρ)·0.9 lie test decisive.
2. Strategy budget goes to movement, not rhetoric: honesty sweeps show the
   verbal layer cannot rescue weak evasion against a belief-driven pursuer.
3. Board-size negotiation is strategic: cop wants 7×7, thief wants bigger.

References: Bernstein et al. (Dec-POMDP complexity); Theraulaz & Bonabeau
(stigmergy); the course rulebook ch. 4–6.

## 4. The RL campaign — an adversarial arms race with promotion gates

Full narrative in the README; every number below is loaded from the committed
experiment artifacts (`results/experiments/`). The campaign's shape: linear
Q-learning → Double-DQN with trap-threat features → two-round arms race via
weight-data crossover between the twin repos → hyperparameter sweep → two
gated promotions, both correctly rejected. Negative results are first-class
artifacts here: they carry the campaign's strongest claims.


In [4]:
linear = json.loads(Path('../results/experiments/rl_training.json').read_text(encoding='utf-8'))
deep = json.loads(Path('../results/experiments/deep_rl_training.json').read_text(encoding='utf-8'))
ens = json.loads(Path('../results/experiments/deep_rl_training_v2_ensemble.json').read_text(encoding='utf-8'))
fine = json.loads(Path('../results/experiments/deep_rl_finetune.json').read_text(encoding='utf-8'))
print('linear: from-scratch vs informed prior curves recorded:', list(linear['runs']))
print('deep v1 vs learned trap cop :', deep['final_survival_vs_deep_cop'])
print('v2 ensemble retrain         :', ens['final_100_game_evals']['vs_learned_trap_cop'], '(collapse, recorded)')
print('fine-tune verdict           :', fine['shipped'])
print('knife-edge                  :', fine['v1_baseline'])


linear: from-scratch vs informed prior curves recorded: ['from_scratch', 'informed_prior']
deep v1 vs learned trap cop : {'win_rate': 1.0, 'games': 100}
v2 ensemble retrain         : 0.06 (collapse, recorded)
fine-tune verdict           : v1 unchanged (best checkpoint was episode 0 - the untouched v1)
knife-edge                  : {'vs_learned_trap_cop': 1.0, 'vs_learned_trap_cop_noisy': 0.0, 'vs_heuristic_trapcop': 1.0}


## 5. The dwell-plateau pin — how much precision does the gate buy?

The book's update rule `tau' = (1-rho)*tau + delta` drives a re-emitted cell to
the fixed point `delta / rho`. With `rho = 0.10` and centre `0.9`, an offset
saturates at the clamp exactly when `delta >= 0.09` — **21 of the 25 kernel
offsets, never the four corners**. So an agent that dwells stamps its own
kernel window on the board, and the emitter can be recovered by fitting that
*shape* back (PRD 10). Per-cell reach-decoding cannot: every saturated cell
decodes to reach 0, so the likelihood ties flat across the plateau.

The pin is gated on Jaccard fit, and the gate is a pure precision/coverage
trade — it never improves both. The sweep below is the sensitivity analysis
behind the shipped value (`PLATEAU_MIN_FIT = 0.9`).


In [ ]:
import json
from pathlib import Path

loc = json.loads(Path("../results/experiments/plateau_localization.json").read_text())
print(f"turns sampled: {loc['turns']}")
print(f"posterior argmax:  exact {loc['argmax_exact_rate']:.3f}"
      f"   mean error {loc['argmax_mean_error']:.2f} cells")
print()
print(f"{'fit gate':>9} {'fires':>7} {'exact':>7} {'mean err':>9}")
for gate, row in loc["by_fit_threshold"].items():
    exact = "-" if row["exact_when_fired"] is None else f"{row['exact_when_fired']:.3f}"
    err = "-" if row["mean_error_when_fired"] is None else f"{row['mean_error_when_fired']:.3f}"
    print(f"{gate:>9} {row['fire_rate']:>7.3f} {exact:>7} {err:>9}")


![Plateau fit sweep](../assets/plateau_fit_sweep.png)

**Reading it.** Precision is nearly flat and very high across the whole range,
while coverage falls steeply — so the gate is cheap insurance rather than a
real trade-off, and the shipped 0.9 buys near-perfect pins on roughly a
quarter of turns. The dotted line is the posterior argmax on the *same* turns:
the pin is not a marginal improvement on it, it answers a question the
posterior cannot (the flat-likelihood tie).

Two caveats worth stating. The argmax baseline here (in-repo evaders) is much
kinder than the 0.07 measured against a live opponent, because these evaders
move more predictably. And a threshold that never fires is not free of risk —
it is the *abstention* that makes a pin safe to consume as near-certainty.


## 6. Trusting an asserted trail — the wire-shape asymmetry

The rulebook calls the scent channel unforgeable, and under the replicated-
engine wire it is: the trail is a *consequence* of applied moves that both
sides recompute. Under the reference (hidden-information) wire it is
**transmitted** — `smell_grid` rides beside `commit`, never inside it — so no
end-of-game hash audit can check it. The property the book relies on is a
property of the *wire shape*, not of scent.

That makes physics the only remaining check: a reading must be explainable by
ONE emitter moving a step per turn. Cells at the clamp outside that envelope
are refused, and the refusal **latches** (re-checking per turn is defeatable,
since a refused turn cannot refresh the anchor).

The honest arm below is the number that matters most: a false positive would
blind the peer for a whole game, which is strictly worse than no check at all.


In [ ]:
forgery = json.loads(Path("../results/experiments/trail_forgery.json").read_text())
print(f"FALSE-POSITIVE RATE (honest traffic): {forgery['false_positive_rate']:.3f}")
print()
outcome = "capture_rate" if "capture_rate" in next(iter(forgery["arms"].values())) else "survival_rate"
print(f"{'arm':>14} {'detected':>9} {outcome:>14} {'refusals':>9}")
for arm, row in forgery["arms"].items():
    print(f"{arm:>14} {row['detected_rate']:>9.3f} {row[outcome]:>14.3f}"
          f" {row['mean_refusals']:>9.2f}")


![Forged trail arms](../assets/trail_forgery_arms.png)

**Reading it.** Every forged arm is now caught, with zero false positives on
honest traffic — including the drifting decoy, which the first version of this
guard could not touch and which was written up as an information-theoretic
limit. That write-up was wrong: the envelope test looks at ONE frame, while the
update law binds consecutive frames. A deposit is non-negative, so no cell may
fall below `(1-rho)` times its own previous value; a decoy that walks its
plateau legally must still teleport its own **history**, and history cannot be
teleported. Detection 0.000 → 1.000, false positives unchanged at 0.000,
validated bit-exactly against a foreign implementation's frames.

What remains open is narrower and honest: a forger who *simulates* a full legal
trail for a fictional trajectory emits frames that satisfy the law, because
they are legal frames — for a different game. Closing that requires binding the
grid to the sealed record, which is a protocol change (ADR-0010), not a check
we can add unilaterally without breaking interop.
